In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
import igraph as ig
from networkx.algorithms import bipartite

In [ ]:
# Load the data
tw_nodes = pd.read_excel('twitter_data_def.xlsx', sheet_name='tw_nodes')
tw_edges = pd.read_excel('twitter_data_def.xlsx', sheet_name='tw_edges')
h_nodes = pd.read_excel('Hyperlink_data_def.xlsx', sheet_name='nodes')
h_edges = pd.read_excel('Hyperlink_data_def.xlsx', sheet_name='edges')

In [ ]:
tw_nodes

In [ ]:
tw_nodes['Category'].unique()

In [ ]:
h_nodes

In [ ]:
seeds_nodes = h_nodes[h_nodes['STATUS']=='IN']

In [ ]:
seeds_nodes['Stance']=[0,0,0,1,1,0,1,0,1,1,1,0,1,1,1,1]

In [ ]:
h_nodes.shape

In [ ]:
h_edges

# Hyperlink

In [ ]:
# create a directed graph among only nodes with 'STATUS' == 'IN'
h_net_seeds = nx.DiGraph()
h_net_sd = nx.DiGraph()
for i in range(h_nodes.shape[0]):
    h_net_sd.add_node(h_nodes['ID'][i])
    if h_nodes['STATUS'][i] == 'IN':
        h_net_seeds.add_node(h_nodes['ID'][i])
for i in range(h_edges.shape[0]):
    if h_edges['Source'][i] in h_net_seeds.nodes() and h_edges['Target'][i] in h_net_seeds.nodes():
        h_net_seeds.add_edge(h_edges['Source'][i], h_edges['Target'][i])
    else:
        h_net_sd.add_edge(h_edges['Source'][i], h_edges['Target'][i])

# remove self loops
h_net_seeds.remove_edges_from(nx.selfloop_edges(h_net_seeds))
#h_net_sd.remove_edges_from(nx.selfloop_edges(h_net_sd))



In [ ]:
h_net_seeds.number_of_nodes(), h_net_seeds.number_of_edges()

In [ ]:
h_net_sd.number_of_nodes(), h_net_sd.number_of_edges()

In [ ]:
# plot h_net_seeds
plt.figure(figsize=(5, 5))
pos = nx.spring_layout(h_net_seeds)
nx.draw(h_net_seeds, pos, node_size=350, node_color='red', alpha=0.7, edge_color='gray', with_labels=True)
plt.show()

In [ ]:
# compute the degree of each node in h_net_seeds
h_net_seeds_indegree = dict(h_net_seeds.in_degree())

# compute the out-degree of each node in h_net_seeds
h_net_seeds_outdegree = dict(h_net_seeds.out_degree())

# histogram of in-degrees
plt.hist(list(h_net_seeds_indegree.values()), bins=9, color='red', alpha=0.7)
plt.xlabel('In-degree')
plt.ylabel('Frequency')
plt.title('Histogram of in-degrees')
plt.show()

# histogram of out-degrees
plt.hist(list(h_net_seeds_outdegree.values()), bins=9, color='red', alpha=0.7)
plt.xlabel('Out-degree')
plt.ylabel('Frequency')
plt.title('Histogram of out-degrees')
plt.show()

In [ ]:

# bar plot of the degree distribution
plt.figure(figsize=(10, 5))
plt.bar(range(len(h_net_seeds_indegree.keys())), h_net_seeds_indegree.values(), color='red', alpha=0.7)
plt.xticks(range(len(h_net_seeds_indegree.keys())), h_net_seeds_indegree.keys())
plt.xlabel('Node ID')
plt.ylabel('In-degree')
plt.title('In-degree distribution of h_net_seeds')
plt.show()


# bar plot of the degree distribution
plt.figure(figsize=(10, 5))
plt.bar(range(len(h_net_seeds_outdegree.keys())), h_net_seeds_outdegree.values(), color='red', alpha=0.7)
plt.xticks(range(len(h_net_seeds_outdegree.keys())), h_net_seeds_outdegree.keys())
plt.xlabel('Node ID')
plt.ylabel('Out-degree')
plt.title('Out-degree distribution of h_net_seeds')
plt.show()

In [ ]:
# show edges of h_net_sd
edgelist = list(h_net_sd.edges())

# Crea un grafo vuoto
B = nx.Graph()

# Aggiungi gli archi dalla edgelist
B.add_edges_from(edgelist)

# Identifica i due insiemi:
# Assumiamo che i primi elementi delle tuple (nodi seed) appartengano a un insieme
# e i secondi elementi (nodi discovered) appartengano all'altro insieme.
seeds = {edge[0] for edge in edgelist}
discovered = {edge[1] for edge in edgelist}

# Assegna l'attributo 'bipartite' ad ogni nodo:
for nodo in seeds:
    B.nodes[nodo]['bipartite'] = 0  # Insieme 0 (ad es. Stakeholders)
for nodo in discovered:
    B.nodes[nodo]['bipartite'] = 1  # Insieme 1 (Organizzazioni o siti esterni)

In [ ]:
h_net_sd.number_of_nodes(), h_net_sd.number_of_edges()

In [ ]:
# Calcola la disposizione dei nodi (layout bipartito)
pos = nx.bipartite_layout(B, seeds)

# Disegna la rete
plt.figure(figsize=(8, 6))
nx.draw(B, pos, with_labels=True, node_color=['skyblue' if B.nodes[n].get('bipartite') == 0 else 'lightgreen' for n in B.nodes()], node_size=150, font_size=10)
plt.title("Rete Bipartita: Stakeholders vs Organizzazioni")
plt.show()

In [ ]:
# compute the degree of each seed
seed_degrees = dict(h_net_sd.degree(seeds))
# bar plot of the degree of each seed
plt.figure(figsize=(8, 6))
plt.bar(range(len(seed_degrees.keys())), seed_degrees.values(), color='skyblue')
plt.xticks(range(len(seed_degrees.keys())), seed_degrees.keys())
plt.xlabel('Seed')
plt.ylabel('Degree')
plt.title('Degree of each seed')
plt.show()

# histogram of the degree of seeds
plt.figure(figsize=(8, 6))
plt.hist(seed_degrees.values(), bins=21, color='skyblue')
plt.xlabel('Degree')
plt.ylabel('Frequency')
plt.title('Degree distribution of seeds')
plt.show()

In [ ]:
# compute the degree of each discovered node
discovered_degrees = dict(h_net_sd.degree(discovered))

# histogram of the degree of discovered nodes
plt.figure(figsize=(8, 6))
plt.hist(discovered_degrees.values(), bins=11, color='lightgreen')
plt.yscale('log')
plt.xlabel('Degree')
plt.ylabel('Frequency')
plt.title(f'Degree distribution of discovered nodes, #nodes={len(discovered_degrees)}')
plt.show()

## Projections

In [ ]:
h_seeds = bipartite.projected_graph(B, seeds)
h_seeds_weighted = bipartite.weighted_projected_graph(B, seeds)

h_discovered = bipartite.projected_graph(B, discovered)
h_discovered_weighted = bipartite.weighted_projected_graph(B, discovered)

In [ ]:
# Visualizza la rete proiettata sui nodi seeds (non ponderata)
plt.figure(figsize=(8, 6))
pos = nx.spring_layout(h_seeds)  # Utilizza un layout spring per la visualizzazione
nx.draw(h_seeds, pos, with_labels=True, node_color='skyblue', node_size=400, font_size=10)
plt.title("Proiezione sui Nodi Seeds")
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
pos = nx.spring_layout(h_seeds_weighted)
edge_widths = [h_seeds_weighted[u][v]['weight'] for u, v in h_seeds_weighted.edges()]
nx.draw(h_seeds_weighted, pos, with_labels=True, node_color='skyblue', 
        node_size=100, font_size=10, width=edge_widths)
plt.title("Proiezione Ponderata sui Nodi Seeds")
plt.show()

In [ ]:
# plot h_net_seeds
plt.figure(figsize=(5, 5))
pos = nx.spring_layout(h_net_seeds)
nx.draw(h_net_seeds, pos, node_size=350, node_color='red', edge_color='gray', with_labels=True)
plt.show()

In [ ]:
h_net_seeds.number_of_nodes(), h_net_seeds.number_of_edges()

In [ ]:
G_filtered.number_of_nodes(), G_filtered.number_of_edges()

In [ ]:
# Soglia desiderata (ad esempio, consideriamo solo archi con weight >= 2)
threshold = 2

# Crea un nuovo grafo filtrato
G_filtered = nx.Graph()

# Aggiungi i nodi dal grafo originale (assicurati di mantenere le informazioni necessarie)
G_filtered.add_nodes_from(h_net_seeds.nodes(data=True))

# Itera sugli archi e aggiungi solo quelli che soddisfano la soglia
for u, v, data in h_seeds_weighted.edges(data=True):
    if data['weight'] >= threshold:
        G_filtered.add_edge(u, v, weight=data['weight'])
plt.figure(figsize=(6, 5))
pos = nx.spring_layout(G_filtered)  # Layout per la visualizzazione
edge_widths = [G_filtered[u][v]['weight'] for u, v in G_filtered.edges()]
nx.draw(G_filtered, pos, with_labels=True, node_color='skyblue',
        node_size=200, font_size=10, width=edge_widths)
plt.title("Proiezione Ponderata Filtrata (threshold >= {})".format(threshold))
plt.show()

In [ ]:
# Soglia desiderata (ad esempio, consideriamo solo archi con weight >= 2)
threshold = 2

# Crea un nuovo grafo filtrato
G_filtered = nx.Graph()

# Aggiungi i nodi dal grafo originale (assicurati di mantenere le informazioni necessarie)
G_filtered.add_nodes_from(h_net_seeds.nodes(data=True))

# Itera sugli archi e aggiungi solo quelli che soddisfano la soglia
for u, v, data in h_seeds_weighted.edges(data=True):
    if data['weight'] >= threshold:
        G_filtered.add_edge(u, v, weight=data['weight'])

edges_G1 = set(h_net_seeds.edges())
edges_G2 = set(G_filtered.edges())

intersection = edges_G1.intersection(edges_G2)
union = edges_G1.union(edges_G2)

jaccard_index = len(intersection) / len(union) if union else 0
print("Jaccard Index:", jaccard_index)


# Converte le matrici di adiacenza in array NumPy
A1 = nx.to_numpy_array(h_net_seeds, nodelist=sorted(h_net_seeds.nodes()))
A2 = nx.to_numpy_array(G_filtered, nodelist=sorted(G_filtered.nodes()))

# Calcola la distanza di Hamming
hamming_distance = np.sum(np.abs(A1 - A2))
print("Hamming Distance:", hamming_distance)


plt.figure(figsize=(6, 5))
pos = nx.spring_layout(G_filtered)  # Layout per la visualizzazione
edge_widths = [G_filtered[u][v]['weight'] for u, v in G_filtered.edges()]
nx.draw(G_filtered, pos, with_labels=True, node_color='skyblue',
        node_size=200, font_size=10, width=edge_widths)
plt.title("Proiezione Ponderata Filtrata (threshold >= {})".format(threshold))
plt.show()

# Twitter

In [ ]:
# Load the data
tw_nodes = pd.read_excel('twitter_data_def.xlsx', sheet_name='tw_nodes')
tw_edges = pd.read_excel('twitter_data_def.xlsx', sheet_name='tw_edges')

In [ ]:
tw_nodes = tw_nodes.merge(h_nodes[['ID', 'STATUS']], left_on='ID', right_on='ID', how='left')

In [ ]:
tw_edges

In [ ]:
tw_edges.groupby('Relationship').size()

In [ ]:
# remove rows with 'Source'=='Target'

tw_edges = tw_edges[tw_edges['Source'] != tw_edges['Target']] 

In [ ]:
tw_edges

In [ ]:
tw_edges.groupby('Relationship').size()

In [ ]:
tw_edges['Date'].hist(bins=460, color='blue', alpha=0.7, figsize=(18, 6))
plt.title(f'Number of Tweets per Week, without self-loops, #tweets={tw_edges.shape[0]}')
plt.xlabel('Date')
plt.ylabel('Number of Tweets')
plt.show()

In [ ]:
tw_edges[tw_edges['Relationship']=='Retweet']['Date'].hist(bins=460, color='blue', alpha=0.7, figsize=(18, 6))
num_retweets = tw_edges[tw_edges['Relationship']=='Retweet'].shape[0]
plt.title(f'Number of Retweets per Week, without self-loops, #retweets={num_retweets}')
plt.xlabel('Date')
plt.ylabel('Number of Tweets')
plt.show()

In [ ]:
tw_net_seeds.number_of_nodes()

In [ ]:
tw_edges.shape

In [ ]:
# create a directed graptw among only nodes with 'STATUS' == 'IN'
tw_net_seeds = nx.DiGraph()
tw_net_sd = nx.DiGraph()
for i in range(tw_nodes.shape[0]):
    tw_net_sd.add_node(tw_nodes['ID'][i])
    if tw_nodes['STATUS'][i] == 'IN':
        tw_net_seeds.add_node(tw_nodes['ID'][i])
for i in range(tw_edges.shape[0]):
    if tw_edges['Source'][i] in tw_net_seeds.nodes() and tw_edges['Target'][i] in tw_net_seeds.nodes():
        tw_net_seeds.add_edge(tw_edges['Source'][i], tw_edges['Target'][i])
    else:
        tw_net_sd.add_edge(tw_edges['Source'][i], tw_edges['Target'][i])

# remove self loops
tw_net_seeds.remove_edges_from(nx.selfloop_edges(tw_net_seeds))
#tw_net_sd.remove_edges_from(nx.selfloop_edges(tw_net_sd))




In [ ]:
# create a directed graptw among only nodes with 'STATUS' == 'IN' only if Relationship == 'Retweet'
rtw_net_seeds = nx.DiGraph()
rtw_net_sd = nx.DiGraph()
for i in range(tw_nodes.shape[0]):
    rtw_net_sd.add_node(tw_nodes['ID'][i])
    if tw_nodes['STATUS'][i] == 'IN':
        rtw_net_seeds.add_node(tw_nodes['ID'][i])
for i in range(tw_edges.shape[0]):
    if tw_edges['Source'][i] in rtw_net_seeds.nodes() and tw_edges['Target'][i] in rtw_net_seeds.nodes() and tw_edges['Relationship'][i] == 'Retweet':
        rtw_net_seeds.add_edge(tw_edges['Source'][i], tw_edges['Target'][i])
    elif tw_edges['Relationship'][i] == 'Retweet':
        rtw_net_sd.add_edge(tw_edges['Source'][i], tw_edges['Target'][i])

# remove self loops
rtw_net_seeds.remove_edges_from(nx.selfloop_edges(rtw_net_seeds))
rtw_net_sd.remove_edges_from(nx.selfloop_edges(rtw_net_sd))




In [ ]:
# 

In [ ]:
tw_net_seeds.number_of_nodes(), tw_net_seeds.number_of_edges()

In [ ]:
# plot tw_net_seeds
plt.figure(figsize=(5, 5))
plt.title('Twitter Network of Seeds')
pos = nx.spring_layout(tw_net_seeds)
nx.draw(tw_net_seeds, pos, node_size=350, node_color='red', edge_color='gray', with_labels=True)
plt.show()

In [ ]:
# plot h_net_seeds with color based on stance
node_colors = []
for i, node in enumerate(tw_net_seeds.nodes()):
    if seeds_nodes.iloc[i]['Stance'] == 1:
        node_colors.append('green')
    elif seeds_nodes.iloc[i]['Stance'] == 0:
        node_colors.append('red')
plt.figure(figsize=(5, 5))
plt.title('Twitter Network of Seeds with Stance towards the Off-Shore Wind Farm')
pos = nx.spring_layout(tw_net_seeds)
nx.draw(tw_net_seeds, pos, node_size=350, node_color=node_colors, edge_color='gray', with_labels=True)
plt.show()


In [ ]:
# plot h_net_seeds with color based on stance
node_colors = []
for i, node in enumerate(rtw_net_seeds.nodes()):
    if seeds_nodes.iloc[i]['Stance'] == 1:
        node_colors.append('green')
    elif seeds_nodes.iloc[i]['Stance'] == 0:
        node_colors.append('red')
plt.figure(figsize=(5, 5))
plt.title('Retwitter Network of Seeds with Stance towards the Off-Shore Wind Farm')
pos = nx.spring_layout(rtw_net_seeds)
nx.draw(rtw_net_seeds, pos, node_size=350, node_color=node_colors, edge_color='gray', with_labels=True,)
plt.show()


In [ ]:
# compute the degree of each node in tw_net_seeds
tw_net_seeds_indegree = dict(tw_net_seeds.in_degree())

# compute the out-degree of each node in tw_net_seeds
tw_net_seeds_outdegree = dict(tw_net_seeds.out_degree())

# histogram of in-degrees
plt.hist(list(tw_net_seeds_indegree.values()), bins=19, color='red', alpha=0.7)
plt.xlabel('In-degree')
plt.ylabel('Frequency')
plt.title('Histogram of in-degrees')
plt.show()

# histogram of out-degrees
plt.hist(list(tw_net_seeds_outdegree.values()), bins=19, color='red', alpha=0.7)
plt.xlabel('Out-degree')
plt.ylabel('Frequency')
plt.title('Histogram of out-degrees')
plt.show()

In [ ]:
# bar plot of the in-degree distribution
plt.figure(figsize=(10, 5))
plt.bar(range(len(tw_net_seeds_indegree.keys())), tw_net_seeds_indegree.values(), color='red', alpha=0.7)
plt.xticks(range(len(tw_net_seeds_indegree.keys())), tw_net_seeds_indegree.keys())
plt.xlabel('Node ID')
plt.ylabel('In-degree')
plt.title('In-degree distribution of tw_net_seeds')
plt.show()

# bar plot of the out-degree distribution
plt.figure(figsize=(10, 5))
plt.bar(range(len(tw_net_seeds_outdegree.keys())), tw_net_seeds_outdegree.values(), color='red', alpha=0.7)
plt.xticks(range(len(tw_net_seeds_outdegree.keys())), tw_net_seeds_outdegree.keys())
plt.xlabel('Node ID')
plt.ylabel('Out-degree')
plt.title('Out-degree distribution of tw_net_seeds')
plt.show()


In [ ]:
tw_net_sd.number_of_nodes(), tw_net_sd.number_of_edges()

In [ ]:
# show edges of h_net_sd
edgelist = list(tw_net_sd.edges())


In [ ]:
# Crea un grafo vuoto
B = nx.Graph()

# Aggiungi gli archi dalla edgelist
B.add_edges_from(edgelist)

# Identifica i due insiemi:
# Assumiamo che i primi elementi delle tuple (nodi seed) appartengano a un insieme
# e i secondi elementi (nodi discovered) appartengano all'altro insieme.
seeds = {edge[0] for edge in edgelist}
discovered = {edge[1] for edge in edgelist}

# Assegna l'attributo 'bipartite' ad ogni nodo:
for nodo in seeds:
    B.nodes[nodo]['bipartite'] = 0  # Insieme 0 (ad es. Stakeholders)
for nodo in discovered:
    B.nodes[nodo]['bipartite'] = 1  # Insieme 1 (Organizzazioni o siti esterni)

In [ ]:
# Calcola la disposizione dei nodi (layout bipartito)
pos = nx.bipartite_layout(B, seeds)

# Disegna la rete
plt.figure(figsize=(8, 6))
nx.draw(B, pos, with_labels=True, node_color=['skyblue' if B.nodes[n].get('bipartite') == 0 else 'lightgreen' for n in B.nodes()], node_size=150, font_size=10)
plt.title("Rete Bipartita: Stakeholders vs Organizzazioni")
plt.show()

In [ ]:
tw_seeds = bipartite.projected_graph(B, seeds)
tw_seeds_weighted = bipartite.weighted_projected_graph(B, seeds)

tw_discovered = bipartite.projected_graph(B, discovered)
tw_discovered_weighted = bipartite.weighted_projected_graph(B, discovered)

In [ ]:
# Visualizza la rete proiettata sui nodi seeds (non ponderata)
plt.figure(figsize=(8, 6))
pos = nx.spring_layout(tw_seeds)  # Utilizza un layout spring per la visualizzazione
nx.draw(tw_seeds, pos, with_labels=True, node_color='skyblue', node_size=400, font_size=10)
plt.title("Proiezione sui Nodi Seeds")
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
pos = nx.spring_layout(tw_seeds_weighted)
edge_widths = [tw_seeds_weighted[u][v]['weight']/10 for u, v in tw_seeds_weighted.edges()]
nx.draw(tw_seeds_weighted, pos, with_labels=True, node_color='skyblue', 
        node_size=150, font_size=10, width=edge_widths)
plt.title("Proiezione Ponderata sui Nodi Seeds")
plt.show()

In [ ]:
# plot h_net_seeds
plt.figure(figsize=(5, 5))
pos = nx.spring_layout(tw_net_seeds)
nx.draw(tw_net_seeds, pos, node_size=350, node_color='red', edge_color='gray', with_labels=True)
plt.show()

In [ ]:
tw_net_seeds.number_of_nodes(), tw_net_seeds.number_of_edges()

In [ ]:
G_filtered.number_of_nodes(), G_filtered.number_of_edges()

In [ ]:
seeds = h_nodes[h_nodes['STATUS']=='IN']
seeds['Stance']=[0,0,0,1,1,0,1,0,1,1,1,0,1,1,1,1]

In [ ]:
# Soglia desiderata (ad esempio, consideriamo solo archi con weight >= 2)
threshold = 1

# Crea un nuovo grafo filtrato
G_filtered = nx.Graph()

# Aggiungi i nodi dal grafo originale (assicurati di mantenere le informazioni necessarie)
G_filtered.add_nodes_from(h_net_seeds.nodes(data=True))

# Itera sugli archi e aggiungi solo quelli che soddisfano la soglia
for u, v, data in h_seeds_weighted.edges(data=True):
    if data['weight'] >= threshold:
        G_filtered.add_edge(u, v, weight=data['weight'])

plt.figure(figsize=(6, 5))
pos = nx.spring_layout(G_filtered)  # Layout per la visualizzazione
edge_widths = [G_filtered[u][v]['weight'] for u, v in G_filtered.edges()]
nx.draw(G_filtered, pos, with_labels=True, node_color=node_colors,
        node_size=200, font_size=10, width=edge_widths)
plt.title("Proiezione Ponderata Filtrata (threshold >= {})".format(threshold))
plt.show()

In [ ]:
# Soglia desiderata (ad esempio, consideriamo solo archi con weight >= 2)
threshold = 6

# Crea un nuovo grafo filtrato
G_filtered = nx.Graph()

# Aggiungi i nodi dal grafo originale (assicurati di mantenere le informazioni necessarie)
G_filtered.add_nodes_from(tw_net_seeds.nodes(data=True))

# Itera sugli archi e aggiungi solo quelli che soddisfano la soglia
for u, v, data in tw_seeds_weighted.edges(data=True):
    if data['weight'] >= threshold:
        G_filtered.add_edge(u, v, weight=data['weight'])

edges_G1 = set(tw_net_seeds.edges())
edges_G2 = set(G_filtered.edges())

intersection = edges_G1.intersection(edges_G2)
union = edges_G1.union(edges_G2)

jaccard_index = len(intersection) / len(union) if union else 0
print("Jaccard Index:", jaccard_index)


# Converte le matrici di adiacenza in array NumPy
A1 = nx.to_numpy_array(tw_net_seeds, nodelist=sorted(tw_net_seeds.nodes()))
A2 = nx.to_numpy_array(G_filtered, nodelist=sorted(G_filtered.nodes()))

# Calcola la distanza di Hamming
hamming_distance = np.sum(np.abs(A1 - A2))
print("Hamming Distance:", hamming_distance)


plt.figure(figsize=(6, 5))
pos = nx.spring_layout(G_filtered)  # Layout per la visualizzazione
edge_widths = [G_filtered[u][v]['weight'] for u, v in G_filtered.edges()]
nx.draw(G_filtered, pos, with_labels=True, node_color='skyblue',
        node_size=200, font_size=10, width=edge_widths)
plt.title("Proiezione Ponderata Filtrata (threshold >= {})".format(threshold))
plt.show()

In [ ]:
# plot the largest connected component

plt.figure(figsize=(8, 6))
largest_cc = max(nx.connected_components(G_filtered), key=len)
G_filtered_lcc = G_filtered.subgraph(largest_cc)
pos = nx.spring_layout(G_filtered_lcc)

node_colors = []
for i, node in enumerate(G_filtered.nodes()):
    if seeds.iloc[i]['Stance'] == 1:
        node_colors.append('green')
    elif seeds.iloc[i]['Stance'] == 0:
        node_colors.append('red')

edge_widths = [G_filtered_lcc[u][v]['weight']/10 for u, v in G_filtered_lcc.edges()]
nx.draw(G_filtered_lcc, pos, with_labels=True, node_color=node_colors,
        node_size=200, font_size=10, width=edge_widths)
plt.title("Largest Connected Component")
plt.show()

In [ ]:
# remove self-loops from tw_net_seeds
tw_net_seeds.remove_edges_from(nx.selfloop_edges(tw_net_seeds))
#plot
plt.figure(figsize=(5, 5))
pos = nx.spring_layout(tw_net_seeds)
nx.draw(tw_net_seeds, pos, node_size=250, node_color='red', edge_color='gray', with_labels=True)
plt.show()


In [ ]:
# plot h_net_seeds
plt.figure(figsize=(5, 5))
pos = nx.spring_layout(tw_net_seeds)
nx.draw(tw_net_seeds, pos, node_size=350, node_color='red', edge_color='gray', with_labels=True)
plt.show()

In [ ]:
jaccard = []
hamming = []

for threshold in range(0, 100):
    # Crea un nuovo grafo filtrato
    G_filtered = nx.Graph()

    # Aggiungi i nodi dal grafo originale (assicurati di mantenere le informazioni necessarie)
    G_filtered.add_nodes_from(tw_net_seeds.nodes(data=True))

    # Itera sugli archi e aggiungi solo quelli che soddisfano la soglia
    for u, v, data in tw_seeds_weighted.edges(data=True):
        if data['weight'] >= threshold:
            G_filtered.add_edge(u, v, weight=data['weight'])

    edges_G1 = set(tw_net_seeds.edges())
    edges_G2 = set(G_filtered.edges())

    intersection = edges_G1.intersection(edges_G2)
    union = edges_G1.union(edges_G2)

    jaccard_index = len(intersection) / len(union) if union else 0
    jaccard.append(jaccard_index)


    # Converte le matrici di adiacenza in array NumPy
    A1 = nx.to_numpy_array(tw_net_seeds, nodelist=sorted(tw_net_seeds.nodes()))
    A2 = nx.to_numpy_array(G_filtered, nodelist=sorted(G_filtered.nodes()))

    # Calcola la distanza di Hamming
    hamming_distance = np.sum(np.abs(A1 - A2))
    hamming.append(hamming_distance)



In [ ]:
jaccard = np.array(jaccard)
hamming = np.array(hamming)


In [ ]:
np.argmax(jaccard), np.max(jaccard)

In [ ]:
plt.figure(figsize=(8, 6))
plt.plot(range(0, 100), jaccard, label="Jaccard Index")
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
plt.plot(range(0, 100), hamming, label="Jaccard Index")
plt.show()

## Retweet

In [ ]:
# Load the data
tw_nodes = pd.read_excel('twitter_data_def.xlsx', sheet_name='tw_nodes')
tw_edges = pd.read_excel('twitter_data_def.xlsx', sheet_name='tw_edges')

In [ ]:
tw_nodes = tw_nodes.merge(h_nodes[['ID', 'STATUS']], left_on='ID', right_on='ID', how='left')

In [ ]:
RT_edges = tw_edges[tw_edges['Relationship']=='Retweet']

In [ ]:
RT_edges = RT_edges[RT_edges['Source'] != RT_edges['Target']]


In [ ]:
RT_edges.reset_index(drop=True, inplace=True)

In [ ]:
RT_edges

In [ ]:
RT_edges['Source'].unique()

In [ ]:
tw_nodes[tw_nodes['STATUS']=='IN']['ID'].unique()

In [ ]:
nodes_missing = set(tw_nodes[tw_nodes['STATUS']=='IN']['ID'].unique()) - set(RT_edges['Source'].unique())

In [ ]:
RT_net_seed = nx.DiGraph()
RT_net_bipa = nx.DiGraph()
for i in range(tw_nodes.shape[0]):
    RT_net_bipa.add_node(tw_nodes['ID'][i])
    if tw_nodes['STATUS'][i] == 'IN':
        RT_net_seed.add_node(tw_nodes['ID'][i])
for i in range(RT_edges.shape[0]):
    if RT_edges['Source'][i] in RT_net_seed.nodes() and RT_edges['Target'][i] in RT_net_seed.nodes():
        RT_net_seed.add_edge(RT_edges['Source'][i], RT_edges['Target'][i])
    else:
        RT_net_bipa.add_edge(RT_edges['Source'][i], RT_edges['Target'][i])

# remove self loops
#RT_net_seed.remove_edges_from(nx.selfloop_edges(RT_net_seed))
#RT_net_bipa.remove_edges_from(nx.selfloop_edges(RT_net_bipa))

In [ ]:
RT_net_seed.number_of_nodes(), RT_net_seed.number_of_edges()

In [ ]:
seeds_nodes = h_nodes[h_nodes['STATUS']=='IN']
seeds_nodes['Stance']=[0,0,0,1,1,0,1,0,1,1,1,0,1,1,1,1]

In [ ]:
plt.figure(figsize=(5, 5))
plt.title('Retweet Network of Seeds with Stance (LCC)')
pos = nx.spring_layout(RT_net_seed)
nx.draw(RT_net_seed, pos, node_size=350, node_color=node_colors, edge_color='gray', with_labels=True)
plt.show()

In [ ]:
node_colors = []
for i, node in enumerate(tw_net_seeds.nodes()):
    if seeds_nodes.iloc[i]['Stance'] == 1:
        node_colors.append('green')
    elif seeds_nodes.iloc[i]['Stance'] == 0:
        node_colors.append('red')
    
seeds_nodes['Color'] = node_colors  

In [ ]:
# plot RT_net_seed
# consider only the largest connected component
largest_cc = max(nx.weakly_connected_components(RT_net_seed), key=len)
RT_net_seed_lcc = RT_net_seed.subgraph(largest_cc)

# nodes removed
nodes_removed = set(RT_net_seed.nodes()) - set(RT_net_seed_lcc.nodes())
# remove nodes_removed from node_colors
node_colors_LCC = [node_colors[i] for i in range(len(node_colors)) if i not in nodes_removed]

In [ ]:
nodes_removed

In [ ]:
seeds_nodes[seeds_nodes['ID'].isin(RT_net_seed_lcc.nodes())]['Color']

In [ ]:
plt.figure(figsize=(5, 5))
plt.title('Retweet Network of Seeds with Stance (LCC)')
pos = nx.spring_layout(RT_net_seed_lcc)
nx.draw(RT_net_seed_lcc, pos, node_size=350, node_color=seeds_nodes[seeds_nodes['ID'].isin(RT_net_seed_lcc.nodes())]['Color'], edge_color='gray', with_labels=True)
plt.show()

In [ ]:
plt.figure(figsize=(5, 5))
plt.title('Retweet Network of Seeds with Stance (LCC)')
pos = nx.spring_layout(RT_net_seed_lcc)
nx.draw(RT_net_seed_lcc, pos, node_size=350, node_color=seeds_nodes[seeds_nodes['ID'].isin(RT_net_seed_lcc.nodes())]['Color'], edge_color='gray', with_labels=True)
plt.show()

In [ ]:
# compute the degree of each node in RT_net_seed
RT_net_seed_indegree = dict(RT_net_seed.in_degree())

# compute the out-degree of each node in RT_net_seed
RT_net_seed_outdegree = dict(RT_net_seed.out_degree())

# histogram of in-degrees
plt.hist(list(RT_net_seed_indegree.values()), bins=19, color='red', alpha=0.7)
plt.xlabel('In-degree')
plt.ylabel('Frequency')     
plt.title('In-degree distribution of RT_net_seed')
plt.show()

# histogram of out-degrees
plt.hist(list(RT_net_seed_outdegree.values()), bins=19, color='red', alpha=0.7)
plt.xlabel('Out-degree')
plt.ylabel('Frequency')
plt.title('Out-degree distribution of RT_net_seed')
plt.show()


In [ ]:
# bar plot of the in-degree distribution
plt.figure(figsize=(10, 5))
plt.bar(range(len(RT_net_seed_indegree.keys())), RT_net_seed_indegree.values(), color=node_colors, alpha=0.7)
plt.xticks(range(len(RT_net_seed_indegree.keys())), RT_net_seed_indegree.keys())
plt.xlabel('Node ID')
plt.ylabel('In-degree')
plt.title('In-degrees of RT_net_seed with Stance')
plt.show()

# bar plot of the out-degree distribution
plt.figure(figsize=(10, 5))
plt.bar(range(len(RT_net_seed_outdegree.keys())), RT_net_seed_outdegree.values(), color=node_colors, alpha=0.7)
plt.xticks(range(len(RT_net_seed_outdegree.keys())), RT_net_seed_outdegree.keys())
plt.xlabel('Node ID')
plt.ylabel('Out-degree')
plt.title('Out-degrees of RT_net_seed with Stance')
plt.show()

In [ ]:
RT_net_bipa.number_of_nodes(), RT_net_bipa.number_of_edges()

In [ ]:
# Crea un grafo vuoto
B = nx.Graph()

# Aggiungi gli archi dalla edgelist
B.add_edges_from(list(RT_net_bipa.edges()))

# Identifica i due insiemi:
# Assumiamo che i primi elementi delle tuple (nodi seed) appartengano a un insieme
# e i secondi elementi (nodi discovered) appartengano all'altro insieme.
seeds = {edge[0] for edge in RT_net_bipa.edges()}
discovered = {edge[1] for edge in RT_net_bipa.edges()}
# Assegna l'attributo 'bipartite' ad ogni nodo:
for nodo in seeds:
    B.nodes[nodo]['bipartite'] = 0  # Insieme 0 (ad es. Stakeholders)
for nodo in discovered:
    B.nodes[nodo]['bipartite'] = 1  # Insieme 1 (Organizzazioni o siti esterni)

In [ ]:
seeds

In [ ]:
# Calcola la disposizione dei nodi (layout bipartito)
pos = nx.bipartite_layout(B, seeds)

# Disegna la rete
plt.figure(figsize=(8, 6))
nx.draw(B, pos, with_labels=True, node_color=['skyblue' if B.nodes[n].get('bipartite') == 0 else 'lightgreen' for n in B.nodes()], node_size=150, font_size=10)
plt.title("Rete Bipartita: Stakeholders vs Organizzazioni")
plt.show()

In [ ]:
nodes_missing

In [ ]:
node_colors_missing

In [ ]:
# Visualizza la rete proiettata sui nodi seeds (non ponderata)
tw_seeds = bipartite.projected_graph(B, seeds)
tw_seeds_weighted = bipartite.weighted_projected_graph(B, seeds)

tw_discovered = bipartite.projected_graph(B, discovered)
tw_discovered_weighted = bipartite.weighted_projected_graph(B, discovered)

node_colors_missing = [node_colors[i] for i in range(len(node_colors)) if i not in nodes_missing]


plt.figure(figsize=(8, 6))
pos = nx.spring_layout(tw_seeds_weighted)
edge_widths = [tw_seeds_weighted[u][v]['weight']/10 for u, v in tw_seeds_weighted.edges()]
nx.draw(tw_seeds_weighted, pos, with_labels=True, node_color=seeds_nodes[seeds_nodes['ID'].isin(tw_seeds.nodes())]['Color'], 
        node_size=150, font_size=10, width=edge_widths)
plt.title("Proiezione Ponderata sui Nodi Seeds")
plt.show()


threshold = 3
# Crea un nuovo grafo filtrato
G_filtered = nx.Graph()
# Aggiungi i nodi dal grafo originale (assicurati di mantenere le informazioni necessarie)
G_filtered.add_nodes_from(RT_net_seed.nodes(data=True))
# Itera sugli archi e aggiungi solo quelli che soddisfano la soglia
for u, v, data in tw_seeds_weighted.edges(data=True):
    if data['weight'] >= threshold:
        G_filtered.add_edge(u, v, weight=1)
# lcc
G_filtered = G_filtered.subgraph(max(nx.connected_components(G_filtered), key=len))

plt.figure(figsize=(8, 6))
pos = nx.spring_layout(G_filtered)  # Layout per la visualizzazione
edge_widths = [G_filtered[u][v]['weight'] for u, v in G_filtered.edges()]
nx.draw(G_filtered, pos, with_labels=True, node_color=seeds_nodes[seeds_nodes['ID'].isin(G_filtered.nodes())]['Color'],
        node_size=200, font_size=10, width=edge_widths)
plt.title("Proiezione Ponderata Filtrata (threshold >= {})".format(threshold))
plt.show()


plt.figure(figsize=(8, 6))
pos = nx.spring_layout(tw_seeds)  # Utilizza un layout spring per la visualizzazione
nx.draw(tw_seeds, pos, with_labels=True, node_color=seeds_nodes[seeds_nodes['ID'].isin(tw_seeds.nodes())]['Color'], node_size=400, font_size=10)
plt.title("Proiezione sui Nodi Seeds")
plt.show()


